# Read the PhoenixZIPReport for sub-101 / ses-102

Reads all raw DICOMs in `sub-101/ses-102`, groups them by `SeriesInstanceUID`, then parses the `PhoenixZIPReport` series (a Structured Report, not a zip file) into a table with every field it contains.

In [4]:
import os
import collections
import pydicom
import pandas as pd

DICOM_DIR = "/mnt/cina/AC/data/sourcedata/dicoms_fl/sub-101/ses-102"

In [ ]:
# Read every file in the session folder and index it by SeriesInstanceUID.
series_files = collections.defaultdict(list)
series_meta = {}

for fname in sorted(os.listdir(DICOM_DIR)):
    fpath = os.path.join(DICOM_DIR, fname)
    if not os.path.isfile(fpath):
        continue
    try:
        ds = pydicom.dcmread(fpath, force=True, stop_before_pixels=True)
    except Exception as exc:
        print(f"Skipping {fname}: {exc}")
        continue
    uid = ds.get("SeriesInstanceUID", None)
    if uid is None:
        continue
    series_files[uid].append(fpath)
    series_meta.setdefault(uid, {
        "SeriesNumber": ds.get("SeriesNumber", ""),
        "SeriesDescription": ds.get("SeriesDescription", ""),
        "ProtocolName": ds.get("ProtocolName", ""),
    })

summary = pd.DataFrame([
    {"SeriesInstanceUID": uid, "NumFiles": len(files), **series_meta[uid]}
    for uid, files in series_files.items()
]).sort_values("SeriesNumber")
summary

## PhoenixZIPReport

Despite the name, the `PhoenixZIPReport` series (SOP Class **Enhanced SR Storage**) is a DICOM Structured Report, not a zip archive to extract. It contains one instance per acquired protocol, each with a `ContentSequence` tree of `TEXT`/`NUM` items (protocol name, sequence name, TR/TE, flip angle, resolution, slice thickness, etc). We walk that tree directly with pydicom rather than unzipping anything.

Note: TR/TE values come out of the SR in microseconds (Siemens SR convention), not the milliseconds you'd see in a normal DICOM header.

In [ ]:
def sr_leaf_values(content_seq, out=None):
    """Flatten a DICOM SR ContentSequence into {ConceptName: value}."""
    if out is None:
        out = {}
    for item in content_seq:
        concept = item.get("ConceptNameCodeSequence", None)
        name = concept[0].get("CodeMeaning", "") if concept else ""
        value_type = item.get("ValueType", "")
        if value_type == "TEXT":
            out[name] = item.TextValue
        elif value_type == "NUM":
            measured = item.get("MeasuredValueSequence", None)
            out[name] = measured[0].get("NumericValue") if measured else None
        if "ContentSequence" in item:
            sr_leaf_values(item.ContentSequence, out)
    return out

In [ ]:
phoenix_uid = next(
    uid for uid, meta in series_meta.items()
    if meta["SeriesDescription"] == "PhoenixZIPReport"
)

rows = []
for fpath in series_files[phoenix_uid]:
    ds = pydicom.dcmread(fpath, force=True)
    values = sr_leaf_values(ds.ContentSequence)
    values["InstanceNumber"] = ds.get("InstanceNumber")
    values["SOPInstanceFile"] = os.path.basename(fpath)
    rows.append(values)

phoenix_df = pd.DataFrame(rows).sort_values("InstanceNumber").reset_index(drop=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
phoenix_df

## Compare all instances of series 10 (`pcasl_3d_multiTI`)

Reads every file belonging to `SeriesInstanceUID = 1.3.12.2.1107.5.2.50.176000.2022110807371176860203590.0.0.0` (17 instances) and reports every top-level header tag whose value differs across instances, so we can see what actually changes from one volume to the next within the series (as opposed to what's constant for the whole series).

In [ ]:
SERIES_10_UID = "1.3.12.2.1107.5.2.50.176000.2022110807371176860203590.0.0.0"

series_10_datasets = [
    pydicom.dcmread(fpath, force=True, stop_before_pixels=True)
    for fpath in series_files[SERIES_10_UID]
]
series_10_datasets.sort(key=lambda ds: int(ds.get("InstanceNumber", 0)))
print(f"{len(series_10_datasets)} instances in series 10")

# Flatten each instance's top-level (non-sequence) tags into {tag: value};
# sequences are recorded as (SQ, item_count) since their contents aren't
# directly comparable as strings.
per_instance_tags = []
all_tags = set()
for ds in series_10_datasets:
    tags = {}
    for elem in ds:
        if elem.tag == (0x7fe0, 0x0010):  # PixelData
            continue
        tags[elem.tag] = ("SQ", len(elem.value)) if elem.VR == "SQ" else elem.value
        all_tags.add(elem.tag)
    per_instance_tags.append(tags)

differing_tags = [
    tag for tag in sorted(all_tags)
    if len({str(t.get(tag, "<MISSING>")) for t in per_instance_tags}) > 1
]
print(f"{len(differing_tags)} tag(s) differ across the 17 instances")

diff_df = pd.DataFrame(
    [
        {"InstanceNumber": ds.get("InstanceNumber"), "File": os.path.basename(ds.filename),
         **{ds[tag].name if tag in ds else str(tag): tags.get(tag, "<MISSING>") for tag in differing_tags}}
        for ds, tags in zip(series_10_datasets, per_instance_tags)
    ]
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
diff_df

### Expand `SharedFunctionalGroupsSequence` and `PerFrameFunctionalGroupsSequence`

These two Enhanced-MR sequences carry per-frame geometry, timing, and ASL labeling state that the top-level tag comparison above skips (it only recorded them as `(SQ, item_count)`). We recursively flatten both sequences into dotted paths (e.g. `Frame03.MR Arterial Spin Labeling Sequence[0].ASL Context`) and diff those paths across the 17 instances the same way. Large embedded blobs (the ~128KB Siemens Phoenix protocol text buried in a private tag) are skipped since they're identical noise, not per-instance signal.

In [ ]:
def flatten_sequence(elem_or_ds, prefix, out, max_len=256):
    """Recursively flatten SQ elements into {dotted_path: value}, skipping PixelData and oversized blobs."""
    for elem in elem_or_ds:
        if elem.tag == (0x7fe0, 0x0010):
            continue
        path = f"{prefix}.{elem.name}" if prefix else elem.name
        if elem.VR == "SQ":
            for i, item in enumerate(elem.value):
                flatten_sequence(item, f"{path}[{i}]", out, max_len)
        else:
            val = elem.value
            if isinstance(val, (bytes, str)) and len(val) > max_len:
                continue
            out[path] = val


# series_10_datasets still has full headers (only PixelData was skipped),
# so SharedFunctionalGroupsSequence / PerFrameFunctionalGroupsSequence are intact.
per_instance_paths = {}
for ds in series_10_datasets:
    flat = {}
    flatten_sequence(ds.SharedFunctionalGroupsSequence[0], "Shared", flat)
    for i, frame in enumerate(ds.PerFrameFunctionalGroupsSequence):
        flatten_sequence(frame, f"Frame{i:02d}", flat)
    per_instance_paths[int(ds.InstanceNumber)] = flat

all_paths = set()
for flat in per_instance_paths.values():
    all_paths.update(flat.keys())

differing_paths = sorted(
    path for path in all_paths
    if len({str(flat.get(path, "<MISSING>")) for flat in per_instance_paths.values()}) > 1
)
print(f"{len(all_paths)} total paths flattened from the two sequences")
print(f"{len(differing_paths)} path(s) differ across the 17 instances")

functional_group_diff_df = pd.DataFrame({
    instance_number: {path: flat.get(path, "<MISSING>") for path in differing_paths}
    for instance_number, flat in sorted(per_instance_paths.items())
})
functional_group_diff_df.index.name = "FunctionalGroupPath"

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
functional_group_diff_df

### (0018,9258) ASL Pulse Train Duration + the real post-labeling-delay schedule

`(0018,9258) ASLPulseTrainDuration` (VR=UL, VM=1) lives inside `ASLSlabSequence`, nested under `MR Arterial Spin Labeling Sequence` in each frame's functional group — it's the duration (ms) of the labeling RF pulse train.

Series 10 has neither `InversionTime` (0018,0082) nor `InversionTimes` (0018,9079) anywhere (confirmed earlier), so despite the `multiTI` protocol name there's no standard DICOM field driving that. The actual timing schedule is buried in the ~128KB Siemens Phoenix protocol text we've been skipping — it's the `alTI[]` array in the raw scanner protocol dump (Siemens reuses this array name from IR sequences; here it holds the pCASL labeling/background-suppression pulse timing, not a per-frame DICOM attribute). We pull it out with a regex, restricted to series 10 only.

In [ ]:
import re

# (0018,9258) ASL Pulse Train Duration - check every frame of every instance
# in series 10 to see whether it actually varies.
pulse_train_durations = set()
for ds in series_10_datasets:
    for frame in ds.PerFrameFunctionalGroupsSequence:
        asl_seq = frame.get((0x0018, 0x9251))
        if asl_seq is None:
            continue
        for slab in asl_seq.value[0][(0x0018, 0x9260)].value:
            pulse_train_durations.add(int(slab[(0x0018, 0x9258)].value))

print(f"(0018,9258) ASL Pulse Train Duration value(s) across all frames/instances: {pulse_train_durations} ms")

# The alTI[] array from the embedded Siemens protocol dump. It's identical
# across all 17 instances (it's the same static protocol text copied into
# each file's SharedFunctionalGroupsSequence), so instance 1 is representative.
blob = (
    series_10_datasets[0]
    .SharedFunctionalGroupsSequence[0][(0x0021, 0x10fe)]
    .value[0][(0x0021, 0x1019)]
    .value.decode("latin1", errors="replace")
)
al_ti = sorted(
    (int(idx), int(val)) for idx, val in re.findall(r"alTI\[(\d+)\]\s*=\s*(\d+)", blob)
)
print(f"{len(al_ti)} alTI[] entries found; distinct values (us): {sorted(set(v for _, v in al_ti))}")
print("alTI[0] (us):", dict(al_ti).get(0))
al_ti[:5]

### Full Siemens Phoenix protocol text (one instance)

Prints the entire embedded XProtocol blob (~128KB) from `SharedFunctionalGroupsSequence[0] -> (0021,10fe) -> (0021,1019)` for a single instance of series 10. It's identical across all 17 instances, so instance 1 is representative of the whole series.

In [ ]:
phoenix_protocol_text = (
    series_10_datasets[0]
    .SharedFunctionalGroupsSequence[0][(0x0021, 0x10fe)]
    .value[0][(0x0021, 0x1019)]
    .value.decode("latin1", errors="replace")
)
print(f"{len(phoenix_protocol_text)} characters")
print(phoenix_protocol_text)



## Series 32 (`ep2d_diff_mddw_20_p2_s3 (0 angle)`) in `sub-102/ses-102`

This is the raw DWI series that made dcm2niix split into two physical outputs during conversion, which then crashed heudiconv's `assert len(bvecs) == len(bvals) == 1`. dcm2niix's own warning was:

> Warning: Series 32 includes partial volume (issue 742): 1 slices acquired but ICE dims (0021,118e) specifies 30

We read every instance's raw header directly (not via the `series_files`/`series_meta` from the top of this notebook, which was built for `sub-101/ses-102`) to see what the `(0021,118e)` ICE dims private tag and slice/instance geometry actually look like across the whole series.

In [ ]:
DICOM_DIR_102_SES102 = "/mnt/cina/AC/data/sourcedata/dicoms_fl/sub-102/ses-102"

series32_files = []
for fname in sorted(os.listdir(DICOM_DIR_102_SES102)):
    fpath = os.path.join(DICOM_DIR_102_SES102, fname)
    if not os.path.isfile(fpath):
        continue
    try:
        ds = pydicom.dcmread(fpath, force=True, stop_before_pixels=True)
    except Exception:
        continue
    if str(ds.get("SeriesNumber", "")) == "32":
        series32_files.append((fpath, ds))

series32_files.sort(key=lambda pair: int(pair[1].get("InstanceNumber", 0)))
print(f"{len(series32_files)} instances found for series 32")

def _tag_value(ds, tag):
    elem = ds.get(tag)
    return elem.value if elem is not None else None

rows = []
for fpath, ds in series32_files:
    rows.append({
        "InstanceNumber": ds.get("InstanceNumber"),
        "SOPInstanceUID": ds.get("SOPInstanceUID"),
        "SliceLocation": ds.get("SliceLocation"),
        "ImagePositionPatient": tuple(ds.get("ImagePositionPatient", [])) if "ImagePositionPatient" in ds else None,
        "ICE Dims (0021,118e)": _tag_value(ds, (0x0021, 0x118e)),
        "NumberOfImagesInMosaic (0019,100a)": _tag_value(ds, (0x0019, 0x100a)),
        "AcquisitionNumber": ds.get("AcquisitionNumber"),
        "DiffusionBValue (0018,9087)": _tag_value(ds, (0x0018, 0x9087)),
        "SoftwareVersions": ds.get("SoftwareVersions"),
        "File": os.path.basename(fpath),
    })

series32_df = pd.DataFrame(rows)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


2263 instances found for series 32


,InstanceNumber,SOPInstanceUID,SliceLocation,ImagePositionPatient,"ICE Dims (0021,118e)","NumberOfImagesInMosaic (0019,100a)",AcquisitionNumber,"DiffusionBValue (0018,9087)",SoftwareVersions,File
0,1,1.3.12.2.1107.5.2.50.176000.30000023012514022537500003829,-40.345900,"(-125.973, -107.867, -40.3459)",b'X_1_1_1_1_1_1_16_1_1_1_1_38_1_1_1_1_73_2_1_30_1_1 ',None,1,None,syngo MR XA31,12192457083.dcm
1,1,1.3.12.2.1107.5.2.50.176000.30000023010405073847900002195,NaN,None,None,None,1,None,syngo MR XA31,12222682671.dcm
2,2,1.3.12.2.1107.5.2.50.176000.30000023012514022537500003830,-35.345900,"(-125.973, -107.867, -35.3459)",b'X_1_1_1_1_1_1_1_1_1_2_1_38_1_1_1_1_73_2_1_30_1_1',None,1,None,syngo MR XA31,12192457084.dcm
3,2,1.3.12.2.1107.5.2.50.176000.30000023010405073847900001791,NaN,None,None,None,2,None,syngo MR XA31,12222682672.dcm
4,3,1.3.12.2.1107.5.2.50.176000.30000023012514022537500003831,-30.345900,"(-125.973, -107.867, -30.3459)",b'X_1_1_1_1_1_1_17_1_1_3_1_38_1_1_1_1_73_2_1_30_1_1 ',None,1,None,syngo MR XA31,12192457085.dcm
5,3,1.3.12.2.1107.5.2.50.176000.30000023010405073847900001789,NaN,None,None,None,3,None,syngo MR XA31,12222682673.dcm
6,4,1.3.12.2.1107.5.2.50.176000.30000023012514022537500003832,-25.345900,"(-125.973, -107.867, -25.3459)",b'X_1_1_1_1_1_1_2_1_1_4_1_38_1_1_1_1_73_2_1_30_1_1',None,1,None,syngo MR XA31,12192457086.dcm
7,4,1.3.12.2.1107.5.2.50.176000.30000023010405073847900001797,NaN,None,None,None,4,None,syngo MR XA31,12222682674.dcm
8,5,1.3.12.2.1107.5.2.50.176000.30000023012514022537500003833,-20.345900,"(-125.973, -107.867, -20.3459)",b'X_1_1_1_1_1_1_18_1_1_5_1_38_1_1_1_1_73_2_1_30_1_1 ',None,1,None,syngo MR XA31,12192457087.dcm
9,5,1.3.12.2.1107.5.2.50.176000.30000023010405073847900001910,NaN,None,None,None,5,None,syngo MR XA31,12222682675.dcm


### Full raw header for one instance of each `SeriesInstanceUID`

The table above showed "series 32" is actually two distinct `SeriesInstanceUID`s sharing the same `SeriesNumber`/`ProtocolName` — one with 73 instances, one with 2190. Print the complete raw header for one representative instance of each so they can be compared directly.

In [6]:
by_uid = {}
for fpath, ds in series32_files:
    uid = str(ds.get("SeriesInstanceUID"))
    by_uid.setdefault(uid, (fpath, ds))

print(f"{len(by_uid)} distinct SeriesInstanceUID(s) found within series 32")

for uid, (fpath, ds) in by_uid.items():
    print("=" * 100)
    print(f"SeriesInstanceUID: {uid}")
    print(f"File: {os.path.basename(fpath)}")
    print("=" * 100)
    print(ds)
    print()

2 distinct SeriesInstanceUID(s) found within series 32
SeriesInstanceUID: 1.3.12.2.1107.5.2.50.176000.30000023012514022537500003827
File: 12192457083.dcm
Dataset.file_meta -------------------------------
(0002, 0000) File Meta Information Group Length  UL: 202
(0002, 0001) File Meta Information Version       OB: b'\x00\x01'
(0002, 0002) Media Storage SOP Class UID         UI: MR Image Storage
(0002, 0003) Media Storage SOP Instance UID      UI: 1.3.12.2.1107.5.2.50.176000.30000023012514022537500003829
(0002, 0010) Transfer Syntax UID                 UI: Implicit VR Little Endian
(0002, 0012) Implementation Class UID            UI: 1.3.12.2.1107.5.2
(0002, 0013) Implementation Version Name         SH: 'SYNGO_MR_XA31A'
(0002, 0016) Source Application Entity Title     AE: 'BRWMR1'
-------------------------------------------------
(0008, 0005) Specific Character Set              CS: 'ISO_IR 100'
(0008, 0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'DIFFUSION', 'NONE